# CIR-Net: Chromosome Image Recognition Network
### TensorFlow/Keras Implementation
Based on the CIR-Net paper with IAI module + InceptionResNetV2 backbone

In [ ]:
# Cell 1 — Install dependencies
!pip install -q tensorflow opencv-python scikit-learn matplotlib seaborn pandas openpyxl

In [ ]:
# Cell 2 — Core imports
import os, glob, re, xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.manifold import TSNE
print('TF version:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# Cell 3 — Configuration
IMG_SIZE    = 224          # resize target
BATCH_SIZE  = 32
EPOCHS_P1   = 20           # frozen backbone
EPOCHS_P2   = 30           # fine-tune top layers
LR          = 0.005
NUM_CLASSES = 24
SEED        = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

# ── UPDATE THIS PATH ──────────────────────────────────────────────
DATA_ROOT   = './Chromo_dataset/Data'
# ─────────────────────────────────────────────────────────────────

IMG_DIR     = os.path.join(DATA_ROOT, 'single_chromosomes_object', 'JEPG')
ANN_DIR     = os.path.join(DATA_ROOT, 'single_chromosomes_object', 'anntations')
TRAIN_LIST  = os.path.join(DATA_ROOT, 'train')
TEST_LIST   = os.path.join(DATA_ROOT, 'test')
CKPT_PATH   = './best_cirnet.keras'
RESULTS_CSV = './results.csv'

# Denver group mapping (paper Table 1)
DENVER = {
    'A': [1,2,3],   'B': [4,5],       'C': list(range(6,13)),
    'D': [13,14,15], 'E': [16,17,18], 'F': [19,20],
    'G': [21,22],    'Sex': [23,24]
}
CLASS_NAMES = [str(i) for i in range(1,23)] + ['X', 'Y']
print('Config OK — Classes:', NUM_CLASSES)

## 1. Dataset Loading

In [ ]:
# Cell 4 — Parse label from filename
# Filename encodes: <patient_id><chr_num><copy>
# e.g. 1101123 → last 2 significant digits before copy = class
# Pattern: filename stem → extract chromosome number (1-24)

def parse_label_from_filename(stem):
    """Extract chromosome class (0-indexed) from filename stem.
    Tries XML annotation first, then falls back to filename heuristic.
    Returns int 0-23 or None if unparseable.
    """
    # Try reading XML annotation
    xml_path = os.path.join(ANN_DIR, stem + '.xml')
    if os.path.exists(xml_path):
        try:
            tree = ET.parse(xml_path)
            root = tree.getroot()
            # Look for <name> tag inside <object>
            for obj in root.findall('object'):
                name_tag = obj.find('name')
                if name_tag is not None:
                    val = name_tag.text.strip()
                    if val.isdigit():
                        cls = int(val)
                        if 1 <= cls <= 24:
                            return cls - 1   # 0-indexed
            # Try attribute <chromosome> at root level
            for tag in ['chromosome', 'label', 'class', 'category']:
                el = root.find(tag)
                if el is not None and el.text:
                    val = el.text.strip()
                    if val.isdigit() and 1 <= int(val) <= 24:
                        return int(val) - 1
        except Exception:
            pass

    # Fallback: filename heuristic
    # strip non-digits, read last 2 digits before final digit as class
    digits = re.sub(r'\D', '', stem)
    if len(digits) >= 2:
        # try last 2 digits
        cls = int(digits[-2:])
        if 1 <= cls <= 24:
            return cls - 1
        # try second-to-last single digit group
        cls = int(digits[-2])
        if 1 <= cls <= 9:
            return cls - 1
    return None

print('Label parser defined')

In [ ]:
# Cell 5 — Load train/test split from provided list files
def read_id_list(path):
    if not os.path.exists(path):
        return []
    with open(path) as f:
        return [ln.strip() for ln in f if ln.strip()]

train_ids = read_id_list(TRAIN_LIST)
test_ids  = read_id_list(TEST_LIST)
print(f'Train IDs: {len(train_ids)} | Test IDs: {len(test_ids)}')

In [ ]:
# Cell 6 — Build records list: (filepath, label_int)
IMG_EXTS = ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff')

def id_to_filepath(img_id):
    """Find image file for a given ID (with or without extension)."""
    stem = str(img_id).strip()
    for ext in IMG_EXTS:
        p = os.path.join(IMG_DIR, stem + ext)
        if os.path.exists(p):
            return p
    # search case-insensitive
    for f in os.listdir(IMG_DIR):
        if os.path.splitext(f)[0] == stem:
            return os.path.join(IMG_DIR, f)
    return None

def build_records(id_list):
    records = []
    for img_id in id_list:
        fp = id_to_filepath(img_id)
        if fp is None:
            continue
        stem = os.path.splitext(os.path.basename(fp))[0]
        label = parse_label_from_filename(stem)
        if label is not None:
            records.append((fp, label))
    return records

# If no list files, fall back to scanning all images in IMG_DIR
if len(train_ids) == 0:
    print('No train/test list found — scanning IMG_DIR directly')
    all_files = [f for f in glob.glob(os.path.join(IMG_DIR, '*')) if f.lower().endswith(IMG_EXTS)]
    all_records = []
    for fp in all_files:
        stem = os.path.splitext(os.path.basename(fp))[0]
        label = parse_label_from_filename(stem)
        if label is not None:
            all_records.append((fp, label))
    train_records, test_records = train_test_split(all_records, test_size=0.15, random_state=SEED,
                                                    stratify=[r[1] for r in all_records])
else:
    train_records = build_records(train_ids)
    test_records  = build_records(test_ids)

print(f'Train records: {len(train_records)} | Test records: {len(test_records)}')
if len(train_records) == 0:
    raise ValueError('No records found! Check DATA_ROOT path and dataset structure.')

In [ ]:
# Cell 7 — Class distribution plot
train_labels = [r[1] for r in train_records]
counts = np.bincount(train_labels, minlength=NUM_CLASSES)
plt.figure(figsize=(14,4))
plt.bar(CLASS_NAMES, counts, color='steelblue')
plt.xlabel('Chromosome Class'); plt.ylabel('Count')
plt.title('Training Set Class Distribution')
plt.xticks(rotation=45)
plt.tight_layout(); plt.show()
print('Total training images:', len(train_records))

In [ ]:
# Cell 8 — Sample image grid
fig, axes = plt.subplots(3, 8, figsize=(16, 6))
shown = set()
idx = 0
for fp, lbl in train_records:
    if lbl not in shown:
        img = cv2.imread(fp, cv2.IMREAD_GRAYSCALE)
        ax = axes[idx // 8][idx % 8]
        ax.imshow(img, cmap='gray')
        ax.set_title(CLASS_NAMES[lbl], fontsize=8)
        ax.axis('off')
        shown.add(lbl)
        idx += 1
    if idx >= 24: break
plt.suptitle('One Sample per Chromosome Class'); plt.tight_layout(); plt.show()

## 2. Preprocessing Pipeline

In [ ]:
# Cell 9 — Preprocessing functions (paper Section 3.1)
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))

def denoise(img):
    """Gaussian denoising."""
    return cv2.GaussianBlur(img, (5,5), 0)

def apply_clahe(img):
    """CLAHE contrast enhancement."""
    return clahe.apply(img)

def otsu_crop(img):
    """Otsu threshold + bounding-box crop."""
    _, mask = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    coords = cv2.findNonZero(mask)
    if coords is None:
        return img
    x,y,w,h = cv2.boundingRect(coords)
    pad = 4
    x1,y1 = max(0,x-pad), max(0,y-pad)
    x2,y2 = min(img.shape[1],x+w+pad), min(img.shape[0],y+h+pad)
    return img[y1:y2, x1:x2]

def preprocess(fp):
    """Full pipeline → (IMG_SIZE, IMG_SIZE, 3) float32 in [0,1]."""
    img = cv2.imread(fp, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.float32)
    img = denoise(img)
    img = apply_clahe(img)
    img = otsu_crop(img)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    return img.astype(np.float32) / 255.0

print('Preprocessing functions defined')

In [ ]:
# Cell 10 — Visualise preprocessing stages
sample_fp = train_records[0][0]
raw  = cv2.imread(sample_fp, cv2.IMREAD_GRAYSCALE)
den  = denoise(raw)
cla  = apply_clahe(den)
crop = otsu_crop(cla)
rsz  = cv2.resize(crop, (IMG_SIZE, IMG_SIZE))

stages = [raw, den, cla, crop, rsz]
titles = ['Raw', 'Denoised', 'CLAHE', 'Otsu Crop', 'Resized']
fig, axes = plt.subplots(1, 5, figsize=(15,3))
for ax, s, t in zip(axes, stages, titles):
    ax.imshow(s, cmap='gray'); ax.set_title(t); ax.axis('off')
plt.suptitle('Preprocessing Pipeline'); plt.tight_layout(); plt.show()

## 3. CDA Augmentation (Paper Algorithm 1)

In [ ]:
# Cell 11 — CDA: Chromosome Data Augmentation
# Applies 23 rotations using rotation matrix A(θ) + random offset b
N_ROTATIONS = 23
ANGLES = np.linspace(0, 360, N_ROTATIONS, endpoint=False)

def cda_augment(img_array):
    """Returns list of N_ROTATIONS augmented images.
    img_array: (H,W,3) float32
    """
    h, w = img_array.shape[:2]
    cx, cy = w / 2, h / 2
    augmented = []
    for theta in ANGLES:
        # Rotation matrix A(θ)
        A = cv2.getRotationMatrix2D((cx, cy), theta, 1.0)
        # Random offset b (paper: small random translation)
        b = np.random.uniform(-0.05 * w, 0.05 * w, size=2)
        A[0, 2] += b[0]
        A[1, 2] += b[1]
        rotated = cv2.warpAffine(img_array, A, (w, h),
                                 flags=cv2.INTER_LINEAR,
                                 borderMode=cv2.BORDER_REFLECT)
        augmented.append(rotated)
    return augmented

print(f'CDA will generate {N_ROTATIONS} augmented copies per training image')

In [ ]:
# Cell 12 — Visualise CDA augmentations
sample_img = preprocess(train_records[0][0])
augs = cda_augment(sample_img)[:8]
fig, axes = plt.subplots(1, 9, figsize=(18,2))
axes[0].imshow(sample_img); axes[0].set_title('Original'); axes[0].axis('off')
for i, a in enumerate(augs):
    axes[i+1].imshow(a); axes[i+1].set_title(f'{ANGLES[i]:.0f}°'); axes[i+1].axis('off')
plt.suptitle('CDA Augmentation Examples'); plt.tight_layout(); plt.show()

## 4. Data Pipeline

In [ ]:
# Cell 13 — Train/Val split + apply CDA to training set
train_sub, val_sub = train_test_split(
    train_records, test_size=0.15, random_state=SEED,
    stratify=[r[1] for r in train_records]
)
print(f'Train: {len(train_sub)} | Val: {len(val_sub)} | Test: {len(test_records)}')

# Build augmented training arrays
print('Preprocessing and augmenting training images...')
X_train_list, y_train_list = [], []
for fp, lbl in train_sub:
    img = preprocess(fp)
    X_train_list.append(img)
    y_train_list.append(lbl)
    for aug in cda_augment(img):
        X_train_list.append(aug)
        y_train_list.append(lbl)

X_train = np.array(X_train_list, dtype=np.float32)
y_train = np.array(y_train_list, dtype=np.int32)
print(f'Augmented train size: {len(X_train)}')

In [ ]:
# Cell 14 — Preprocess val and test sets (no augmentation)
print('Preprocessing val/test...')
X_val  = np.array([preprocess(fp) for fp,_ in val_sub],    dtype=np.float32)
y_val  = np.array([lbl          for _,lbl in val_sub],     dtype=np.int32)
X_test = np.array([preprocess(fp) for fp,_ in test_records], dtype=np.float32)
y_test = np.array([lbl          for _,lbl in test_records],  dtype=np.int32)
print(f'Val: {X_val.shape} | Test: {X_test.shape}')

In [ ]:
# Cell 15 — One-hot encode + class weights
from sklearn.utils.class_weight import compute_class_weight

y_train_oh = tf.keras.utils.to_categorical(y_train, NUM_CLASSES)
y_val_oh   = tf.keras.utils.to_categorical(y_val,   NUM_CLASSES)
y_test_oh  = tf.keras.utils.to_categorical(y_test,  NUM_CLASSES)

cls_weights_arr = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=y_train)
class_weight_dict = dict(enumerate(cls_weights_arr))
print('Class weights computed (Y chromosome gets higher weight)')

In [ ]:
# Cell 16 — Build tf.data pipelines
AUTOTUNE = tf.data.AUTOTUNE

train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train_oh))\
    .shuffle(len(X_train), seed=SEED).batch(BATCH_SIZE).prefetch(AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val_oh))\
    .batch(BATCH_SIZE).prefetch(AUTOTUNE)

test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test_oh))\
    .batch(BATCH_SIZE).prefetch(AUTOTUNE)

print(f'Batches — train: {len(train_ds)} | val: {len(val_ds)} | test: {len(test_ds)}')

## 5. CIR-Net Model Architecture

In [ ]:
# Cell 17 — IAI Module (Iterative Attention Integration, paper Eq. 7-9)
class IAIModule(layers.Layer):
    """Channel-wise attention: squeeze → excite → scale."""
    def __init__(self, channels, reduction=16, **kwargs):
        super().__init__(**kwargs)
        self.gap  = layers.GlobalAveragePooling2D(keepdims=True)
        self.fc1  = layers.Dense(max(channels // reduction, 1), activation='relu')
        self.fc2  = layers.Dense(channels, activation='sigmoid')

    def call(self, x):
        # Eq. 7: global context
        s = self.gap(x)                   # (B,1,1,C)
        s = tf.squeeze(s, axis=[1,2])     # (B,C)
        # Eq. 8-9: attention weights
        a = self.fc1(s)
        a = self.fc2(a)
        a = tf.reshape(a, (-1, 1, 1, tf.shape(x)[-1]))
        return x * a

print('IAI module defined')

In [ ]:
# Cell 18 — Build CIR-Net
def build_cirnet(num_classes=NUM_CLASSES, img_size=IMG_SIZE, trainable_backbone=False):
    inputs = keras.Input(shape=(img_size, img_size, 3))

    # Backbone: InceptionResNetV2 pretrained on ImageNet
    backbone = keras.applications.InceptionResNetV2(
        include_top=False, weights='imagenet',
        input_tensor=inputs
    )
    backbone.trainable = trainable_backbone

    x = backbone.output                  # feature maps
    x = IAIModule(x.shape[-1], name='iai')(x)  # attention
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = Model(inputs, outputs, name='CIRNet')
    return model, backbone

model, backbone = build_cirnet()
model.summary(line_length=80)

In [ ]:
# Cell 19 — Compile model
model.compile(
    optimizer=keras.optimizers.RMSprop(learning_rate=LR),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
print('Model compiled — RMSProp lr=', LR)

## 6. Training

In [ ]:
# Cell 20 — Callbacks
callbacks = [
    keras.callbacks.ModelCheckpoint(CKPT_PATH, monitor='val_accuracy',
                                     save_best_only=True, verbose=1),
    keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=10,
                                   restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                       patience=5, min_lr=1e-6, verbose=1)
]
print('Callbacks defined')

In [ ]:
# Cell 21 — Phase 1: Train with frozen backbone
print('=== Phase 1: Frozen backbone ===')
hist1 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=EPOCHS_P1, callbacks=callbacks,
    class_weight=class_weight_dict
)

In [ ]:
# Cell 22 — Phase 2: Fine-tune top 50 backbone layers
print('=== Phase 2: Fine-tuning top 50 layers ===')
backbone.trainable = True
for layer in backbone.layers[:-50]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.RMSprop(learning_rate=LR / 10),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

hist2 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=EPOCHS_P2, callbacks=callbacks,
    class_weight=class_weight_dict
)

In [ ]:
# Cell 23 — Plot training curves
acc  = hist1.history['accuracy']     + hist2.history['accuracy']
val  = hist1.history['val_accuracy'] + hist2.history['val_accuracy']
loss = hist1.history['loss']         + hist2.history['loss']
vloss= hist1.history['val_loss']     + hist2.history['val_loss']
ep   = range(1, len(acc)+1)
ep_p2 = len(hist1.history['accuracy'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14,4))
ax1.plot(ep, acc, label='Train Acc'); ax1.plot(ep, val, label='Val Acc')
ax1.axvline(ep_p2, color='gray', linestyle='--', label='Fine-tune start')
ax1.set_title('Accuracy'); ax1.legend()
ax2.plot(ep, loss, label='Train Loss'); ax2.plot(ep, vloss, label='Val Loss')
ax2.axvline(ep_p2, color='gray', linestyle='--')
ax2.set_title('Loss'); ax2.legend()
plt.tight_layout(); plt.show()

## 7. Evaluation

In [ ]:
# Cell 24 — Load best checkpoint and evaluate on test set
best_model = keras.models.load_model(CKPT_PATH, custom_objects={'IAIModule': IAIModule})
test_loss, test_acc = best_model.evaluate(test_ds, verbose=0)
print(f'Test Accuracy: {test_acc*100:.2f}%  |  Test Loss: {test_loss:.4f}')

In [ ]:
# Cell 25 — Per-class classification report
y_pred_prob = best_model.predict(test_ds)
y_pred = np.argmax(y_pred_prob, axis=1)
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

In [ ]:
# Cell 26 — Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(16,12))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            cmap='Blues')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.title('Confusion Matrix — Raw Counts')
plt.tight_layout(); plt.show()

In [ ]:
# Cell 27 — Normalised confusion matrix
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
plt.figure(figsize=(16,12))
sns.heatmap(cm_norm, annot=True, fmt='.2f', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            cmap='YlOrRd', vmin=0, vmax=1)
plt.xlabel('Predicted'); plt.ylabel('True')
plt.title('Confusion Matrix — Normalised')
plt.tight_layout(); plt.show()

In [ ]:
# Cell 28 — Per-class accuracy bar chart
per_cls_acc = cm_norm.diagonal()
plt.figure(figsize=(14,4))
bars = plt.bar(CLASS_NAMES, per_cls_acc*100, color='teal')
plt.axhline(test_acc*100, color='red', linestyle='--', label=f'Overall {test_acc*100:.1f}%')
plt.xlabel('Class'); plt.ylabel('Accuracy (%)')
plt.title('Per-Class Accuracy'); plt.legend()
plt.xticks(rotation=45); plt.tight_layout(); plt.show()

In [ ]:
# Cell 29 — ROC curves (one-vs-rest)
from sklearn.metrics import roc_curve, auc
plt.figure(figsize=(10,8))
for i in range(NUM_CLASSES):
    if y_test_oh[:, i].sum() == 0: continue
    fpr, tpr, _ = roc_curve(y_test_oh[:, i], y_pred_prob[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, lw=1, label=f'Chr {CLASS_NAMES[i]} (AUC={roc_auc:.2f})')
plt.plot([0,1],[0,1],'k--')
plt.xlabel('FPR'); plt.ylabel('TPR')
plt.title('ROC Curves — One vs Rest')
plt.legend(bbox_to_anchor=(1.05,1), loc='upper left', fontsize=7)
plt.tight_layout(); plt.show()

## 8. Qualitative Analysis

In [ ]:
# Cell 30 — T-SNE feature embedding (paper Fig. 4c)
feat_model = Model(inputs=best_model.input,
                   outputs=best_model.layers[-3].output)  # before final dense

MAX_TSNE = 1000
idx_ts = np.random.choice(len(X_test), min(MAX_TSNE, len(X_test)), replace=False)
feats = feat_model.predict(X_test[idx_ts], verbose=0)
labels_ts = y_test[idx_ts]

print('Running T-SNE...')
tsne = TSNE(n_components=2, random_state=SEED, perplexity=30, n_iter=1000)
emb = tsne.fit_transform(feats)

plt.figure(figsize=(10,8))
sc = plt.scatter(emb[:,0], emb[:,1], c=labels_ts, cmap='tab20', s=15, alpha=0.8)
plt.colorbar(sc, ticks=range(NUM_CLASSES), label='Chromosome Class')
plt.title('T-SNE Feature Embedding'); plt.tight_layout(); plt.show()

In [ ]:
# Cell 31 — GradCAM heatmap
import tensorflow.keras.backend as K

def gradcam(model, img_array, class_idx):
    """Returns overlaid GradCAM heatmap."""
    # Find last conv layer
    last_conv = [l for l in model.layers if isinstance(l, layers.Conv2D)][-1]
    grad_model = Model(model.inputs, [last_conv.output, model.output])

    with tf.GradientTape() as tape:
        inp = tf.cast(img_array[np.newaxis], tf.float32)
        conv_out, preds = grad_model(inp)
        loss = preds[:, class_idx]

    grads = tape.gradient(loss, conv_out)
    pooled = tf.reduce_mean(grads, axis=[1,2])[0]
    cam = tf.reduce_sum(conv_out[0] * pooled, axis=-1).numpy()
    cam = np.maximum(cam, 0)
    cam = cv2.resize(cam, (IMG_SIZE, IMG_SIZE))
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    heatmap = cv2.applyColorMap(np.uint8(255*cam), cv2.COLORMAP_JET)
    overlay = cv2.addWeighted(np.uint8(img_array*255), 0.6, heatmap, 0.4, 0)
    return overlay

# Show GradCAM for 4 test samples
fig, axes = plt.subplots(1, 4, figsize=(16,4))
for i, ax in enumerate(axes):
    img = X_test[i]
    cls = y_test[i]
    cam_img = gradcam(best_model, img, cls)
    ax.imshow(cv2.cvtColor(cam_img, cv2.COLOR_BGR2RGB))
    ax.set_title(f'Chr {CLASS_NAMES[cls]}')
    ax.axis('off')
plt.suptitle('GradCAM Attention Maps'); plt.tight_layout(); plt.show()

## 9. Comparison Baselines

In [ ]:
# Cell 32 — Helper to train a baseline model quickly
def train_baseline(backbone_fn, name, epochs=15):
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    base = backbone_fn(include_top=False, weights='imagenet', input_tensor=inputs)
    base.trainable = False
    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.Dropout(0.2)(x)
    out = layers.Dense(NUM_CLASSES, activation='softmax')(x)
    m = Model(inputs, out, name=name)
    m.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    h = m.fit(train_ds, validation_data=val_ds, epochs=epochs, verbose=0)
    _, acc = m.evaluate(test_ds, verbose=0)
    print(f'{name}: {acc*100:.2f}%')
    return name, acc

results = [('CIR-Net (Ours)', test_acc)]
print('Training comparison models...')

In [ ]:
# Cell 33 — Train ResNet50, EfficientNetB3, MobileNetV3
results.append(train_baseline(keras.applications.ResNet50, 'ResNet-50'))
results.append(train_baseline(keras.applications.EfficientNetB3, 'EfficientNet-B3'))
results.append(train_baseline(keras.applications.MobileNetV3Small, 'MobileNetV3'))

# Comparison bar chart
names, accs = zip(*results)
colors = ['gold'] + ['steelblue']*(len(results)-1)
plt.figure(figsize=(8,5))
plt.bar(names, [a*100 for a in accs], color=colors)
plt.ylabel('Test Accuracy (%)'); plt.title('Model Comparison')
plt.xticks(rotation=20, ha='right'); plt.tight_layout(); plt.show()

## 10. Cross-Validation

In [ ]:
# Cell 34 — 5-Fold Stratified Cross-Validation (paper Section 4.4.1)
# NOTE: This trains 5 models — may take time. Set SKIP_CV=True to skip.
SKIP_CV = False

if not SKIP_CV:
    # Use original (non-augmented) train data for CV
    X_cv = np.array([preprocess(fp) for fp,_ in train_records], dtype=np.float32)
    y_cv = np.array([lbl for _,lbl in train_records], dtype=np.int32)
    y_cv_oh = tf.keras.utils.to_categorical(y_cv, NUM_CLASSES)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    cv_scores = []
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_cv, y_cv)):
        print(f'\n--- Fold {fold+1}/5 ---')
        m, _ = build_cirnet()
        m.compile(optimizer=keras.optimizers.RMSprop(LR),
                  loss='categorical_crossentropy', metrics=['accuracy'])
        fold_ds = tf.data.Dataset.from_tensor_slices(
            (X_cv[tr_idx], y_cv_oh[tr_idx])).shuffle(len(tr_idx)).batch(BATCH_SIZE).prefetch(AUTOTUNE)
        vfold_ds = tf.data.Dataset.from_tensor_slices(
            (X_cv[va_idx], y_cv_oh[va_idx])).batch(BATCH_SIZE).prefetch(AUTOTUNE)
        m.fit(fold_ds, validation_data=vfold_ds, epochs=EPOCHS_P1, verbose=0)
        _, acc = m.evaluate(vfold_ds, verbose=0)
        cv_scores.append(acc)
        print(f'Fold {fold+1} Accuracy: {acc*100:.2f}%')

    print(f'\nCV Accuracy: {np.mean(cv_scores)*100:.2f}% ± {np.std(cv_scores)*100:.2f}%')
    print('Paper reports: 96.99% ± 1.70%')
else:
    print('CV skipped (SKIP_CV=True)')

## 11. Denver Group Analysis & Abnormality Detection

In [ ]:
# Cell 35 — Denver group accuracy
group_results = {}
for grp, chrs in DENVER.items():
    # 0-indexed: chr 1-24 → 0-23
    idxs = [i for i,c in enumerate(y_test) if (c+1) in chrs]
    if len(idxs) == 0: continue
    correct = sum(y_test[i] == y_pred[i] for i in idxs)
    group_results[grp] = correct / len(idxs)

plt.figure(figsize=(8,4))
plt.bar(group_results.keys(), [v*100 for v in group_results.values()], color='mediumpurple')
plt.ylabel('Accuracy (%)'); plt.title('Accuracy by Denver Group')
plt.tight_layout(); plt.show()
for g,v in group_results.items():
    print(f'Group {g}: {v*100:.1f}%')

In [ ]:
# Cell 36 — Abnormality detection (low-confidence flagging)
CONF_THRESHOLD = 0.70
max_conf = y_pred_prob.max(axis=1)
flagged = np.where(max_conf < CONF_THRESHOLD)[0]
print(f'Samples flagged as potentially abnormal (conf < {CONF_THRESHOLD}): {len(flagged)}')

if len(flagged) > 0:
    fig, axes = plt.subplots(1, min(4, len(flagged)), figsize=(14,3))
    if len(flagged) == 1: axes = [axes]
    for ax, i in zip(axes, flagged[:4]):
        ax.imshow(X_test[i])
        ax.set_title(f'True:{CLASS_NAMES[y_test[i]]}\nConf:{max_conf[i]:.2f}', fontsize=9)
        ax.axis('off')
    plt.suptitle('Low-Confidence Predictions (Possible Abnormalities)')
    plt.tight_layout(); plt.show()

## 12. Export & Inference

In [ ]:
# Cell 37 — Save results CSV
df = pd.DataFrame({
    'true_class':      [CLASS_NAMES[i] for i in y_test],
    'predicted_class': [CLASS_NAMES[i] for i in y_pred],
    'confidence':      max_conf,
    'correct':         (y_test == y_pred)
})
df.to_csv(RESULTS_CSV, index=False)
print(f'Results saved to {RESULTS_CSV}')
print(df.head())

In [ ]:
# Cell 38 — Export SavedModel and TFLite
best_model.save('./cirnet_saved_model')
print('SavedModel saved.')

converter = tf.lite.TFLiteConverter.from_saved_model('./cirnet_saved_model')
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()
with open('./cirnet_quantized.tflite', 'wb') as f:
    f.write(tflite_model)
print(f'TFLite model saved ({len(tflite_model)/1e6:.1f} MB)')

In [ ]:
# Cell 39 — Single image inference demo
def predict_chromosome(image_path, model):
    img = preprocess(image_path)
    probs = model.predict(img[np.newaxis], verbose=0)[0]
    cls = np.argmax(probs)
    conf = probs[cls]
    print(f'Predicted: Chromosome {CLASS_NAMES[cls]}  ({conf*100:.1f}% confidence)')
    plt.figure(figsize=(4,4))
    plt.imshow(img)
    plt.title(f'Chr {CLASS_NAMES[cls]} — {conf*100:.1f}%')
    plt.axis('off'); plt.show()
    return CLASS_NAMES[cls], conf

# Demo on first test image
demo_fp = test_records[0][0]
predict_chromosome(demo_fp, best_model)

In [ ]:
# Cell 40 — Final summary table
from IPython.display import display
summary = pd.DataFrame(results, columns=['Model', 'Test Accuracy'])
summary['Test Accuracy (%)'] = (summary['Test Accuracy']*100).round(2)
summary = summary.drop('Test Accuracy', axis=1)
summary['Params'] = ['~55M (CIR-Net)', '~25M', '~12M', '~4M']
display(summary)
print(f'\nBest Model: CIR-Net | Test Acc: {test_acc*100:.2f}%')